In [356]:
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import re
import time

In [416]:
#helper function
def clean_municipality(name):
    name = re.sub(r"^\d{4}\s", "", name)
    name = name.replace(" (bostad)", "")
    name = name.replace(" (arbetsställe)", "")
    return name
    
def filter_on_vals (df, column, values):
    result = df[ df[column].isin(values) ]
    return result

def time_to_minutes(time_str):
    hours, minutes, seconds = map(int, time_str.split(":"))
    return hours * 60 + minutes

def minutes_to_time(minutes):
    hours = minutes // 60
    mins = minutes % 60
    return f"{hours:02d}:{mins:02d}:00"

def get_time_samples (earliest, latest, optimal, alpha, beta, n_samples = 1):
    arrival_minutes = np.arange(time_to_minutes(earliest), time_to_minutes(latest))
    mins_to_optimal = time_to_minutes(optimal) - arrival_minutes
    weights         = np.exp(-beta * np.abs(mins_to_optimal) - alpha * np.maximum(-mins_to_optimal, 0))
    probabilities   = weights / weights.sum()
    return np.random.choice(arrival_minutes, p=probabilities, size = n_samples)

def get_relevant_trips(train_schedules, origin, destination):
    """Return trip IDs that visit origin before destination.(LLM function for efficiency) """

    # Keep only rows at the two relevant stops
    stops = train_schedules.loc[train_schedules['stop_name'].isin([origin, destination]),['trip_id', 'stop_name', 'stop_sequence']]

    # Get the first occurrence of each stop in each trip
    stops = (stops.sort_values('stop_sequence').drop_duplicates(['trip_id', 'stop_name']))

    # Put origin and destination sequences side by side
    sequences = stops.pivot(index='trip_id', columns='stop_name', values='stop_sequence')

    # Keep trips visiting both stops, in the correct order
    valid = sequences.dropna(subset=[origin, destination])
    valid = valid[valid[destination] > valid[origin]]

    if valid.empty:
        raise ValueError("No valid trips found for this route!")
    return valid.index.to_numpy()

In [ ]:
# def get_relevant_trips ( train_schedules , origin, destination ):
#     stops = train_schedules.loc[train_schedules['stop_name'].isin([origin, destination]),['trip_id', 'stop_name', 'stop_sequence']]
#     stops = (stops.sort_values('stop_sequence').drop_duplicates(['trip_id', 'stop_name']))
#     print(stops)
#     assert 1==0
#     trips = train_trips["trip_id"].unique()
#     for trip in trips:
#         this_trip = train_schedules[train_schedules["trip_id"] == trip]
#         relevant  = set([origin,destination]).issubset(set(this_trip['stop_name'].dropna())) #checks if the trip contains the relevant stops
#         if relevant:
#             orig_sequence = this_trip.loc[this_trip['stop_name'] == origin, 'stop_sequence'].iloc[0]
#             dest_sequence = this_trip.loc[this_trip['stop_name'] == destination, 'stop_sequence'].iloc[0]
#             if dest_sequence < orig_sequence: # train going the other way
#                 trips = trips[trips != trip]  # remove trip  
#         else: trips = trips[trips != trip]  # remove trip if not relevant
#     if  len(trips) == 0:
#         raise ExceptionError("No valid trips found for this route!") 
#     return trips

# def get_train_time (train_schedules, origin, destination, arrival_time, target_type, alpha = 1.0, beta = 1.0):
#     """
#     arrival_time: should be the planned arrival time of the person at the destination
#     target_type: either "arrive", "leave" 
#     alpha: penalty for trains leaving after arrival_time
#     beta: penalty for trains that is leaving further away from arrival_time
#     --->> returns the time of the train that the person will likely take
#     """
#     reduced_trips = filter_on_vals(train_schedules, 'stop_name', [origin, destination])
#     trip_id = get_relevant_trips (reduced_trips , origin, destination)

#     relevant_trips = filter_on_vals(train_schedules, 'trip_id', trip_id)
#     trains_to_dest = filter_on_vals(relevant_trips, 'stop_name', [destination])
#     times_to_dest  = trains_to_dest['arrival_time'].values
#     mins_to_target = arrival_time - times_to_dest
#     weights        = np.exp(-beta * np.abs(mins_to_target) - alpha * np.maximum(-mins_to_target, 0))
#     probabilities  = weights / weights.sum()
#     arrival_time   = np.random.choice(times_to_dest, p=probabilities)
    
#     match target_type:
#         case "arrive": #arrive tells you the 'outflow' ie how many people would be at the station that are coming out of a train
#             time = arrival_time 
#         case "leave" : #leave tells you the 'intflow' ie how many people would be at the station to take a train
#             trip_id = relevant_trips.loc[(relevant_trips['arrival_time'] == arrival_time) & (relevant_trips['stop_name'] == destination)
#                                          , 'trip_id'].iloc[0]
#             trip_to_take = filter_on_vals(relevant_trips, 'trip_id', [trip_id])
#             departure_time = trip_to_take.loc[trip_to_take['stop_name'] == origin, 'departure_time'].iloc[0]
#             time = departure_time
#         case _:
#             raise ValueError("either 'arrive' or 'leave'")
#     return time

## Preparing the Train Schedule Data

This section processes General Transit Feed Specification (GTFS) data for train services in Skåne, Sweden. It filters the original datasets to include selected train operators: 'Öresundståg','Pågatåg','PågatågExpress'. 

**Load the GTFS datasets**
* `routes`: Contains information about transit routes, including route descriptions and route IDs.
* `trips`: Contains individual trips and their associated route IDs and service IDs.
* `times`: Contains stop-by-stop schedules for individual trips, including arrival times, departure times, and stop sequences.
* `stops`: Contains information about stations and stops, including stop IDs and names.
* `calendar`: Contains service calendar information, including service IDs and the dates during which services are scheduled to operate.

The `arrival_time` and `departure_time` columns are converted using the custom `time_to_minutes()` function. The function transforms time strings into numerical values representing minutes, making the times easier to compare and use in calculations. For example, `08:35:00` becomes `515`, representing 515 minutes after midnight. This representation is useful for comparing train times.

The `train_sched` DataFrame is reduced to the following columns:

| Column           | Description                                           |
| ---------------- | ----------------------------------------------------- |
| `trip_id`        | Unique identifier for the train trip.                 |
| `stop_sequence`  | Order in which the stop occurs during the trip.       |
| `arrival_time`   | Scheduled arrival time, expressed in minutes.         |
| `departure_time` | Scheduled departure time, expressed in minutes.       |
| `start_date`     | Start date of the associated service calendar period. |
| `end_date`       | End date of the associated service calendar period.   |
| `stop_name`      | Name of the station or stop.                          |

The final `train_sched` DataFrame serves as the prepared train schedule for the simulation.

In [274]:
folder = Path("skane")

routes = pd.read_csv(folder/"routes.csv")
trips  = pd.read_csv(folder/"trips.csv")
times  = pd.read_csv(folder/"stop_times.csv")
stops  = pd.read_csv(folder/"stops.csv")
calendar = pd.read_csv(folder/"calendar.csv")

train_companies = ['Öresundståg','Pågatåg','PågatågExpress'] #use routes["route_desc"].unique() to identify train companies

train_routes = filter_on_vals(routes, "route_desc", train_companies) 
route_id = train_routes["route_id"] #dtype: str

train_trips = filter_on_vals(trips, "route_id", route_id.values)
trip_id = train_trips["trip_id"] #dtype: int

train_sched = filter_on_vals(times, "trip_id", trip_id)

# Create a lookup dictionary for the dates
start_dates = calendar.set_index('service_id')['start_date']
end_dates = calendar.set_index('service_id')['end_date']
service_ids = trips.set_index('trip_id')['service_id']
# Add dates to train_sched
train_sched["start_date"] = (train_sched['trip_id'].map(service_ids)).map(start_dates)
train_sched["end_date"] = (train_sched['trip_id'].map(service_ids)).map(end_dates)

# Create a lookup dictionary for the stops
train_stops = stops.set_index('stop_id')['stop_name']
# Add to train_sched
train_sched['stop_name'] = train_sched['stop_id'].map(train_stops)

#Replace times into comparable int (24hr military time)
train_sched['arrival_time'] = train_sched['arrival_time'].apply(time_to_minutes)
train_sched['departure_time'] = train_sched['departure_time'].apply(time_to_minutes)

#Select relevant columns
train_sched = train_sched[['trip_id','stop_sequence','arrival_time','departure_time','start_date', 'end_date', 'stop_name']]

stations = list(train_sched["stop_name"].unique())

## Preparing the Worker and Commuter Data

This section processes employment statistics for municipalities in Skåne, Sweden. The data is filtered to include the selected year and individual genders, municipality names are standardized, and railway stations are assigned to each worker's residential and workplace municipalities. The columns are renamed to English to make the dataset easier to work with throughout the simulation.

| Original column                         | New column                 | Description                                                             |
| --------------------------------------- | -------------------------- | ----------------------------------------------------------------------- |
| `kön`                                   | `sex`                      | Gender category.                                                        |
| `bostadskommun`                         | `residential_municipality` | Municipality where the worker lives.                                    |
| `arbetsställekommun`                    | `workplace_municipality`   | Municipality where the worker works.                                    |
| `år`                                    | `year`                     | Year to which the statistics apply.                                     |
| `Sysselsatta 15-74 år. Årligt register` | `employed_people`          | Number of employed people aged 15–74, according to the annual register. |


The variable `year` is set to `2024`, defining the year of employment statistics used in the simulation. Each worker record is assigned two stations using the `municipality_to_station` mapping. This mapping was created by an LLM, where we fed the list of municipalities and asked it to pick the closest station. Some municipalities may not have a corresponding station in the mapping - these are removed.

In [417]:
folder = Path("skane")
csv_file = folder / "TAB1830_sv.csv"

all_workers = pd.read_csv( csv_file, sep=",", dtype=str, encoding="cp1252")
all_workers = all_workers[["kön", "bostadskommun", "arbetsställekommun", "år", 'Sysselsatta 15-74 år. Årligt register']].rename(columns={
                            "kön": "sex",
                            "bostadskommun": "residential_municipality",
                            "arbetsställekommun": "workplace_municipality",
                            "år": "year",
                            'Sysselsatta 15-74 år. Årligt register': "employed_people"})

year = 2024 #<------------- select year here!

#We want the current data and seperate genders
all_workers = all_workers[(all_workers["year"]==str(year)) & (all_workers["sex"]!="totalt")]
all_workers["residential_municipality"] = (all_workers["residential_municipality"].apply(clean_municipality))
all_workers["workplace_municipality"] = (all_workers["workplace_municipality"].apply(clean_municipality))

#assign the stations
all_workers["origin_station"] = (all_workers["residential_municipality"].map(municipality_to_station))
all_workers["destination_station"] = (all_workers["workplace_municipality"].map(municipality_to_station))

all_workers = all_workers[all_workers["origin_station"].notna() & all_workers["destination_station"].notna()]
#all_workers.to_csv( Path("skane") / "all_workers_2024.csv", index=False, encoding="utf-8-sig")

## Generating Commuters

### `get_commuters()`
The function identifies workers whose origin or destination station is the target station. Each employed person is then expanded into an individual commuter record with a unique `worker_id`. A desired arrival time is randomly generated for each commuter using the specified time range and probability parameters.

**Inputs:**
* `all_workers`: Employment data, including municipality, gender, station assignments, and employment counts.
* `target_station`: Station being studied in the simulation.
* `earliest`, `latest`, `optimal`: Earliest, latest, and preferred commuter arrival times.
* `alpha`, `beta`: Parameters controlling the weighted probability distribution used to generate desired arrival times.

In [418]:
def get_commuters (all_workers, target_station, earliest = "08:00:00", latest = "09:15:00", optimal = "08:45:00", 
                   alpha = 2.0, beta = 1.0):
    """
    alpha: additional penalty for trains arriving after optimal
    beta: penalty for distance from optimal
    """
    
    target_workers = all_workers[ (all_workers["origin_station"] == target_station) | (all_workers["destination_station"] == target_station) ] 
    target_workers["employed_people"] = target_workers["employed_people"].astype(int)
    target_workers = target_workers[target_workers["employed_people"] != 0] 

    #We take only workers that would commute
    all_commuters = target_workers[ target_workers["origin_station"] != target_workers["destination_station"] ]
    commuters = all_commuters.loc[all_commuters.index.repeat(all_commuters["employed_people"])].copy()

    # Assign a unique ID to each commuter
    commuters = commuters.reset_index(drop=True)
    commuters["worker_id"] = range(1, len(commuters) + 1)
    
    commuters["desired_arrival"] = get_time_samples(earliest, latest, optimal, alpha, beta, n_samples = len(commuters))
    commuters["desired_arrival"] = commuters["desired_arrival"].apply(minutes_to_time)
    commuters["commute_type"] = [["arrive","leave"][i] for i in (commuters["destination_station"] != target_station).astype(int)]
    commuters = commuters[ ['worker_id','sex','residential_municipality','workplace_municipality','origin_station','destination_station',
                           'desired_arrival', 'commute_type'] ]        
    return commuters

## Selecting Train Times

### `get_train_time()`
The function filters the schedule to the origin and destination stations and identifies trips that serve both stations in the correct order. It then calculates a selection probability for each valid train based on how close its destination arrival time is to the commuter's desired arrival time. A train is randomly selected according to these probabilities. If `target_type` is `arrive`, the function returns its arrival time at the destination. If `target_type` is `leave`, it returns the departure time of that same train from the origin.

**Inputs:**
* `train_schedules`: Prepared train schedule containing trip IDs, station names, and arrival and departure times.
* `origin`, `destination`: Stations between which the commuter travels.
* `arrival_time`: The commuter's desired arrival time, represented numerically in minutes.
* `target_type`: Specifies whether to return the train's arrival time (`arrive`) or departure time (`leave`).
* `alpha`, `beta`: Parameters controlling the probability of selecting each train based on its arrival time relative to the desired time.

In [370]:
def get_train_time(train_schedules, origin, destination, arrival_time, target_type, alpha=1.0, beta=1.0):
    """
    (LLM function for efficiency)
    alpha: additional penalty for trains arriving after arrival_time
    beta: penalty for distance from arrival_time
    """

    if target_type not in ('arrive', 'leave'):
        raise ValueError("target_type must be 'arrive' or 'leave'")

    relevant_stops = train_schedules.loc[train_schedules['stop_name'].isin([origin, destination])]
    trip_ids = get_relevant_trips(relevant_stops, origin, destination)
    relevant = relevant_stops[relevant_stops['trip_id'].isin(trip_ids)]

    # Get destination arrivals and their corresponding trip IDs
    arrivals = relevant.loc[relevant['stop_name'].eq(destination),['trip_id', 'arrival_time']].dropna(subset=['arrival_time'])

    if arrivals.empty:
        raise ValueError("No valid destination arrival times found!")

    # Calculate selection probabilities
    times = arrivals['arrival_time'].to_numpy()
    difference = arrival_time - times

    log_weights = (-beta * np.abs(difference)-alpha * np.maximum(-difference, 0))
    weights = np.exp(log_weights - log_weights.max())
    probabilities = weights / weights.sum()

    # Select a row, keeping its trip ID linked to its arrival time
    selected_idx = np.random.choice(len(arrivals), p=probabilities)
    selected = arrivals.iloc[selected_idx]

    if target_type == 'arrive':
        return selected['arrival_time']

    # Retrieve the departure time for the same trip at origin
    origin_departures = relevant.loc[relevant['stop_name'].eq(origin)  ,  ['trip_id', 'departure_time']]
    departures = origin_departures.loc[origin_departures['trip_id'].eq(selected['trip_id'])  , 'departure_time']

    if departures.empty:
        raise ValueError("No departure time found for selected trip!")

    return departures.iloc[0]

## Simulating Commuter Behaviour
The function filters the train schedule to include only services active on the specified date, then generates individual commuters and their desired arrival times. For each commuter, it selects a likely train based on the desired arrival time and the train-selection probability parameters. Failed commuter simulations are excluded.

It then generates waiting times for commuters departing from the target station, calculates when they will be present at that station, and converts the simulated times into `HH:MM:SS` format.

**Inputs**
* `date`: Date for which the simulation is run.
* `train_sched`: Train schedule containing trips and their service dates.
* `workers`: Employment data with residential and workplace municipalities and assigned stations.
* `target_station`: Station being studied.
* `earliest_arrival`, `latest_arrival`, `optimal_arrival`: Time boundaries and preferred arrival time for commuters.
* `hyperparams`: Dictionary containing probability distribution parameters for desired arrival times, train selection, and waiting times, as well as minimum, maximum, and average waiting times.
* `plot`: If `True`, displays plots of simulated time distributions.
* `testing`: Optional station used to restrict the simulation to commuters travelling to or from that station.
* `save`: If `True`, saves the resulting commuter data to a CSV file.

In [402]:
def simulate (date, train_sched, workers, target_station, earliest_arrival, latest_arrival, optimal_arrival, 
              hyperparams, plot=False, testing = None, save=True):
    
    active_trains = train_sched[ (train_sched["start_date"] <= date) & (train_sched["end_date"] >= date) ]
    
    if active_trains.empty:
        raise ValueError(f"No active train trips on {date}")

    com_alpha = hyperparams["commuter_dist_alpha"]
    com_beta  = hyperparams["commuter_dist_beta"]
    
    commuters = get_commuters(workers, target_station, earliest = earliest_arrival, latest = latest_arrival, 
                              optimal = optimal_arrival, alpha = com_alpha, beta = com_beta)

    if plot:
        plt.figure(figsize=(12,5))
        pd.Series(commuters['desired_arrival']).value_counts().plot(kind="bar")
        plt.title("Distribution of Desired Arrival Times")
        plt.xlabel("Arrival Times")
        plt.ylabel("Frequency")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

    if testing is not None:
        to_Testing = commuters[commuters["destination_station"] == testing]
        from_Testing = commuters[commuters["origin_station"] == testing]
        commuters = pd.concat([to_Testing,from_Testing], ignore_index=True)   

    n_commuters = len(commuters)
    train_schedules = []
    train_alpha = hyperparams["train_dist_alpha"]
    train_beta  = hyperparams["train_dist_beta"]
    failed_indices = []
    
    for origin, destination, arrival_time, commute_type, i, in zip (commuters["origin_station"], commuters["destination_station"], 
                                                                    commuters["desired_arrival"], commuters["commute_type"], 
                                                                    range(len(commuters))):
        try:
            start = time.perf_counter()
            time_in_min = get_train_time(active_trains, origin, destination, time_to_minutes(arrival_time),
                                         commute_type, alpha = train_alpha, beta = train_beta)
            train_schedules.append(time_in_min)
            end = time.perf_counter()
            speed = end-start
            print(f"Simulation Progress: {int(100 * (i+1)/n_commuters): 3} %, Estimated Time Left : {int(speed*(n_commuters-(i+1))) :4} s, Persons Left: {n_commuters-(i+1) : 6} "
                  , end="\r", flush =True)
        except Exception as e:
            failed_indices.append(i)
            print(f"Commuter {i} failed: {e}" + "                                                                                      " , end="\r", flush =True)
    
    commuters = commuters.drop(index=failed_indices)
        
    wait_alpha = hyperparams["wait_dist_alpha"]
    wait_beta  = hyperparams["wait_dist_beta"]

    min_wait_time = hyperparams["min_wait_time"]
    max_wait_time = hyperparams["max_wait_time"]
    avg_wait_time = hyperparams["avg_wait_time"]
    
    commuters["train_to_take"] = train_schedules
    commuters["waiting_time"]  = get_time_samples(min_wait_time, max_wait_time, avg_wait_time, wait_alpha , wait_beta , n_samples = len(commuters)) * commuters["commute_type"].eq("leave").astype(int)
    commuters["in " + target_station] = commuters["train_to_take"] - commuters["waiting_time"]

    commuters["train_to_take"] = commuters["train_to_take"].apply(minutes_to_time)
    commuters["waiting_time"]  = commuters["waiting_time"].apply(minutes_to_time)
    commuters["in " + target_station] = commuters["in " + target_station].apply(minutes_to_time)
    
    if testing is not None and plot:
        plt.figure(figsize=(12,5))
        pd.Series(commuters[commuters["commute_type"] == "arrive"]["train_to_take"]).value_counts().plot(kind="bar")
        plt.title("Distribution of Arrival Times from "+target_station)
        plt.xlabel("Train Arrival Times")
        plt.ylabel("Frequency")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(12,5))
        pd.Series(commuters[commuters["commute_type"] == "leave"]["train_to_take"]).value_counts().plot(kind="bar")
        plt.title("Distribution of Departure Times from "+target_station)
        plt.xlabel("Train Departure Times")
        plt.ylabel("Frequency")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(12,5))
        pd.Series(commuters["waiting_time"]).value_counts().plot(kind="bar")
        plt.title("Distribution of Waiting Times in "+target_station)
        plt.xlabel("Train Waiting Times")
        plt.ylabel("Frequency")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

    if save:
        title = target_station.replace(" ", "_") + "_" + str(date) +"_all_commuters.csv"
        commuters.to_csv( Path("skane") / title, index=False, encoding="utf-8-sig")
        
    return commuters

### Run the simulation!

Run this cell to generate a file with simulated values. Set testing to a station name to reduce the number of workers to be simulated. This also allows us to see the distribution for the train times, arrival times, and waiting times (assuming that plot = True). We can use these distributions to see adjust the hyperparameters.

In [420]:
date = 20261014  #Format year-month-day, dtype = int
schedule = train_sched #Prepared pd dataframe
workers  = all_workers #Prepared pd dataframe
target_station   = "Lund C" #See the list stations for all possible station names
earliest_arrival = "08:30:00" # min time when the worker would arrive at the station (its working municipality station)
latest_arrival   = "09:10:00" # max time when the worker would arrive at the station (its working municipality station)
optimal_arrival  = "08:45:00" # ideal time to arrive
hyperparams = {'commuter_dist_alpha': 0.9, 
               'commuter_dist_beta': 0.03,
               'train_dist_alpha': 2.0,
               'train_dist_beta' : 0.1,
               'wait_dist_alpha': 2.0,
               'wait_dist_beta' : .01,
               'avg_wait_time': "00:10:00",
               'min_wait_time': "00:01:00",
               'max_wait_time': "00:30:00"}

testing = None #"Landskrona station" #Change to a station name for testing
commuters = simulate (date, schedule, workers, target_station, earliest_arrival, latest_arrival, optimal_arrival, 
              hyperparams, plot=False, testing = testing, save = True)

Simulation Progress:  100 %, Estimated Time Left :    0 s, Persons Left:      0 